# vLLM Adapter Image Smoke Test

This notebook sends a vision request to the local vLLM API using the `spine_adapter` LoRA module.

In [ ]:
import base64
import json
import mimetypes
from pathlib import Path

import requests

image_path = Path('/home/jovyan/llm_train/img/0aba1c3b-IMG_3566_spine_028.png')
if not image_path.exists():
    raise FileNotFoundError(f'Image not found: {image_path}')

mime_type, _ = mimetypes.guess_type(str(image_path))
mime_type = mime_type or 'image/png'
image_b64 = base64.b64encode(image_path.read_bytes()).decode('utf-8')
image_data_url = f'data:{mime_type};base64,{image_b64}'

payload = {
    'model': 'spine_adapter',
    'messages': [
        {
            'role': 'user',
            'content': [
                {
                    'type': 'text',
                    'text': 'Extract book info from the spine image. Return strict JSON with keys: title, author, call_no.'
                },
                {
                    'type': 'image_url',
                    'image_url': {'url': image_data_url}
                }
            ]
        }
    ],
    'max_tokens': 256,
    'temperature': 0.0
}

resp = requests.post('http://127.0.0.1:8000/v1/chat/completions', json=payload, timeout=300)
print('status:', resp.status_code)
resp.raise_for_status()
data = resp.json()
print(json.dumps(data, indent=2)[:4000])

if data.get('choices'):
    print('\nModel text output:\n')
    print(data['choices'][0]['message']['content'])